# 12 · Model C: small LLM that writes the location code (GPU)

**Idea (after ReaGeo, Alibaba Amap 2026):** fine-tune a small language model so that, given an address, it *writes* a location code character by character. The code is hierarchical, so each further correct character shrinks the error.

**Output format:** DIGIPIN, India Post's 10-character grid code (each character picks one of 16 sub-squares; 10 characters is about 4 m). Set `FMT` to `'geohash'` or `'coords'` to compare formats.

**Retrieval (on by default):** places from the public gazetteer whose names appear in the address are added to the prompt with their codes:

```
Address: opp Ratnadeep Supermarket, Madhapur
Known places: Ratnadeep Supermarket = 422-5M7-FPFM; Madhapur = 422-5M6-2K93
-> 422-5M7-FPL4
```

The model can copy a prefix and adjust it. That is its way into cold areas, where it has seen no training addresses.

**Training:** supervised fine-tuning (SFT) with LoRA adapters (only about 1% of weights are trained). This is stage 1. Stage 2, reinforcement learning with a distance reward (GRPO), is a separate notebook that starts from the adapter saved here; it is only worth building once this stage's numbers are in.

**Model:** `Qwen/Qwen2.5-0.5B-Instruct` first (Apache-2.0, fast), then `Qwen/Qwen2.5-1.5B-Instruct` if it helps.

**Runtime:** about 40 minutes on a T4 for 100,000 addresses, 2 epochs, with the 0.5B model. **Needs:** notebooks 04 to 06. Checked on CPU with a tiny random network; not yet trained on a GPU.

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz', 'transformers>=4.44', 'peft', 'scikit-learn']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/content/Navigate' if os.path.exists('/content') else '/kaggle/working/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

import torch
assert torch.cuda.is_available() or P['where'] == 'local', 'No GPU: Runtime > Change runtime type > T4 GPU (Colab) or Settings > Accelerator (Kaggle)'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU trial run)')

In [ ]:
from bench import baselines as bl, context as cx, io, metrics as mt
from models import common, llm_geocoder as lg
common.set_seed(7)
MODEL, FMT, USE_RETRIEVAL = 'Qwen/Qwen2.5-0.5B-Instruct', 'digipin', True
TRAIN_ROWS, EPOCHS = 100_000, 2
train, val, test = common.load_bench(P['bench'], TRAIN_ROWS)
print(f"{len(train):,} training addresses | example target: {lg.encode_target(train.lat[0], train.lon[0], FMT)}")

## Retrieval context from the public gazetteer
Same gazetteer the rules baseline uses: public landmarks only, so hidden landmarks stay hidden.

In [ ]:
ctx_train = ctx_test = None
if USE_RETRIEVAL:
    G, B = P['gaz'], P['bench']
    lm = pd.read_parquet(f'{B}/landmarks_clean.parquet')
    roads = cx.load_roads(G); area_poly, localities, suburbs = cx.load_areas(G)
    gg = bl.GazetteerGeocoder(bl.build_gazetteer(lm, area_poly, localities, suburbs, roads))
    ctx_train, ctx_test = lg.make_context(gg, train, FMT), lg.make_context(gg, test, FMT)
    print('share of training rows with at least one known place:', round(float(np.mean([bool(c) for c in ctx_train])), 3))
    print('example:', train.text[0], '|', ctx_train[0])

## Fine-tune

In [ ]:
model, tok = lg.load_model(MODEL)
print(lg.build_prompt(tok, train.text[0], FMT, ctx_train[0] if ctx_train else None))
lg.train_sft(model, tok, train, fmt=FMT, context=ctx_train, epochs=EPOCHS, batch_size=16, lr=2e-4)

## Predict and save
`raw` is exactly what the model wrote. Rows where it wrote something that is not a valid code count as no answer.

In [ ]:
NAME = 'llm_sft' + ('' if USE_RETRIEVAL else '_noretrieval') + ('' if FMT == 'digipin' else f'_{FMT}')
pred = lg.predict(model, tok, test, fmt=FMT, context=ctx_test)
io.save_run(P['runs'], NAME, pred, about=f'{MODEL} fine-tuned with LoRA to write the {FMT} code' + (', with gazetteer retrieval in the prompt.' if USE_RETRIEVAL else '.'))
_, res = mt.evaluate(test, pred, NAME, slices=False)
print(mt.leaderboard(res).to_string())
print(pred[['raw', 'lat', 'lon']].head(8).to_string())
print(f"valid codes: {pred.lat.notna().mean():.1%} | speed: {pred.latency_ms.median():.1f} ms per address (batched)")
model.save_pretrained(f"{P['runs']}/{NAME}/adapter")          # the GRPO stage starts from this